# RTB ≡ Trust-PCL: A Practical Demonstration

**Deleu et al. (2025)** proved that Relative Trajectory Balance (RTB) — a GFlowNet
training objective — is mathematically equivalent to Trust-PCL, an off-policy RL
method with KL regularization ([arXiv:2509.01632](https://arxiv.org/abs/2509.01632)).

This notebook demonstrates the equivalence empirically: we train **two completely
independent GFlowNets** — one using RTB, one using Trust-PCL — from cloned
initializations, and show they produce **identical trajectories, losses, and
learned distributions** at every training step.

### Parameter correspondence

| RTB (GFlowNet) | Trust-PCL (RL) | Relationship |
|:---|:---|:---|
| `beta` (reward scaling) | `alpha` (temperature) | `alpha = 1/beta` |
| `logZ` (log partition) | `v_soft_s0` (soft value at s₀) | `v_soft_s0 = alpha * logZ` |
| `pf` (posterior policy) | `policy` | Same role |
| `prior_pf` (prior policy) | `reference_policy` | Same role |
| `L_RTB` | `L_Trust-PCL` | `L_TPCL = α² · L_RTB` |

### Key insight: the α² scaling is just a learning rate rescaling

Since Trust-PCL loss = α² · RTB loss, the gradients differ by a factor of α².
To get **identical parameter updates**, we compensate by setting:

$$\text{lr}_{\text{Trust-PCL}} = \frac{\text{lr}_{\text{RTB}}}{\alpha^2}$$

We use `beta=2.0` / `alpha=0.5` so α²=0.25 and the scaling is clearly visible.

In [7]:
import matplotlib.pyplot as plt
import torch

from gfn.estimators import DiscretePolicyEstimator
from gfn.gflownet import RelativeTrajectoryBalanceGFlowNet, TrustPCLGFlowNet
from gfn.gym import HyperGrid
from gfn.preprocessors import KHotPreprocessor
from gfn.samplers import Sampler
from gfn.utils.common import set_seed
from gfn.utils.modules import DiscreteUniform, MLP
from gfn.utils.training import validate

## 1. Environment and shared hyperparameters

We use a small 2D HyperGrid (8×8 = 64 states) with `calculate_partition=True` so we can compare the learned distribution to the ground truth.

In [ ]:
SEED = 42
BETA = 2.0                     # RTB reward scaling
ALPHA = 1.0 / BETA             # Trust-PCL temperature = 0.5
LR_RTB = 5e-4                  # RTB learning rate
LR_TPCL = LR_RTB / ALPHA**2    # Compensated: lr / α² = 1e-3 / 0.25 = 4e-3
N_ITERATIONS = 2000
BATCH_SIZE = 128
HIDDEN_DIM = 64

print(f"beta={BETA}, alpha={ALPHA}, alpha^2={ALPHA**2}")
print(f"lr_rtb={LR_RTB}, lr_tpcl={LR_TPCL}")
print(f"lr_tpcl / lr_rtb = {LR_TPCL / LR_RTB} (should be 1/alpha^2 = {1/ALPHA**2})")

# Environment (shared — both GFlowNets train on the same env).
env = HyperGrid(
    ndim=2,
    height=8,
    calculate_partition=True,
    store_all_states=True,
)
preprocessor = KHotPreprocessor(height=env.height, ndim=env.ndim)
print(f"\nHyperGrid: {env.height}^{env.ndim} = {env.n_states} states")

beta=2.0, alpha=0.5, alpha^2=0.25
lr_rtb=0.0001, lr_tpcl=0.0004
lr_tpcl / lr_rtb = 4.0 (should be 1/alpha^2 = 4.0)

HyperGrid: 8^2 = 64 states


## 2. Build two independent GFlowNets from cloned weights

We create the RTB GFlowNet first, then build the Trust-PCL GFlowNet with
**separate** neural networks whose weights are cloned from RTB. This ensures
they start identical but are trained independently — no shared state.

### About the prior (reference policy) and what logZ estimates

RTB learns a posterior `p_post(x) ∝ p_prior(x) · r(x)^β`. The learned `logZ`
converges to `log Σ_x p_prior(x) · r(x)^β`, which depends on the prior.

We use a **uniform prior** (equal probability for all *actions* at every state).
Crucially, uniform over actions ≠ uniform over states: the exit action fires
at every step, so short trajectories are far more likely. The origin gets ~33%
of prior mass while the far corner gets ~0.2%.

This means `logZ` converges to a value much smaller than the environment's raw
`log Σ r(x)` — because the prior downweights states far from the origin. This
is correct: `logZ` estimates the normalizing constant of the *prior-tilted*
target, not the raw reward.

**The RTB ↔ Trust-PCL equivalence holds regardless of the prior choice.**

In [19]:
def make_posterior(preprocessor, n_actions, hidden_dim):
    """Build a trainable DiscretePolicyEstimator with an MLP."""
    module = MLP(
        input_dim=preprocessor.output_dim,
        output_dim=n_actions,
        hidden_dim=hidden_dim,
    )
    return DiscretePolicyEstimator(
        module=module,
        n_actions=n_actions,
        preprocessor=preprocessor,
        is_backward=False,
    )


def make_uniform_prior(preprocessor, n_actions):
    """Build a fixed uniform prior (no learnable parameters)."""
    return DiscretePolicyEstimator(
        module=DiscreteUniform(n_actions),
        n_actions=n_actions,
        preprocessor=preprocessor,
        is_backward=False,
    )


# --- RTB GFlowNet ---
set_seed(SEED)
pf_rtb = make_posterior(preprocessor, env.n_actions, HIDDEN_DIM)
prior_rtb = make_uniform_prior(preprocessor, env.n_actions)
gfn_rtb = RelativeTrajectoryBalanceGFlowNet(
    pf=pf_rtb, prior_pf=prior_rtb, beta=BETA, init_logZ=0.0,
)

# --- Trust-PCL GFlowNet (cloned posterior, same uniform prior) ---
pf_tpcl = make_posterior(preprocessor, env.n_actions, HIDDEN_DIM)
prior_tpcl = make_uniform_prior(preprocessor, env.n_actions)
# Clone posterior weights so both start identical.
pf_tpcl.load_state_dict(pf_rtb.state_dict())

gfn_tpcl = TrustPCLGFlowNet(
    policy=pf_tpcl,
    reference_policy=prior_tpcl,
    alpha=ALPHA,
    init_v_soft_s0=ALPHA * 0.0,  # = alpha * init_logZ = 0.0
)

# Verify: initial posterior weights are identical.
for (n1, p1), (n2, p2) in zip(
    pf_rtb.named_parameters(), pf_tpcl.named_parameters()
):
    assert torch.equal(p1, p2), f"Mismatch in {n1}"
print("Initial posterior weights are identical between RTB and Trust-PCL.")
print(f"Prior: uniform (DiscreteUniform, no learnable parameters)")

Initial posterior weights are identical between RTB and Trust-PCL.
Prior: uniform (DiscreteUniform, no learnable parameters)


## 3. Train both independently

At each step we:
1. Reset the RNG to the same seed → both GFlowNets sample **identical trajectories**
2. Compute and backprop loss independently
3. Log losses, logZ, v_soft_s0, and the weight difference norm

The learning rates are set so that `lr_tpcl = lr_rtb / α²`, which compensates
for the α² loss scaling and produces identical gradient updates.

In [20]:
# Separate optimizers with compensated learning rates.
# We use SGD (not Adam) because Adam's adaptive step size depends on
# gradient magnitude, and the alpha^2 scaling would interact with
# Adam's second-moment normalization in a way that breaks exact
# equivalence. With plain SGD, update = lr * grad, so the alpha^2
# in the gradient cancels exactly with 1/alpha^2 in the learning rate.
opt_rtb = torch.optim.SGD(
    list(gfn_rtb.pf.parameters()) + gfn_rtb.logz_parameters(), lr=LR_RTB,
)
opt_tpcl = torch.optim.SGD(
    list(gfn_tpcl.pf.parameters()) + gfn_tpcl.logz_parameters(), lr=LR_TPCL,
)

# Separate samplers (each uses its own posterior policy).
sampler_rtb = Sampler(estimator=pf_rtb)
sampler_tpcl = Sampler(estimator=pf_tpcl)

# Logging.
log = {"loss_rtb": [], "loss_tpcl": [], "logZ": [], "v_soft_s0": [], "weight_diff": []}

for step in range(N_ITERATIONS):
    # --- RTB step ---
    torch.manual_seed(SEED + step)  # Same seed for both.
    trajs_rtb = sampler_rtb.sample_trajectories(env, n=BATCH_SIZE, save_logprobs=True)
    opt_rtb.zero_grad()
    loss_rtb = gfn_rtb.loss(env, trajs_rtb)
    loss_rtb.backward()
    opt_rtb.step()

    # --- Trust-PCL step (same seed → identical trajectories) ---
    torch.manual_seed(SEED + step)
    trajs_tpcl = sampler_tpcl.sample_trajectories(env, n=BATCH_SIZE, save_logprobs=True)
    opt_tpcl.zero_grad()
    loss_tpcl = gfn_tpcl.loss(env, trajs_tpcl)
    loss_tpcl.backward()
    opt_tpcl.step()

    # --- Log ---
    with torch.no_grad():
        logZ_val = gfn_rtb.logZ.item()
        v_soft_val = gfn_tpcl.v_soft_s0.item()
        weight_diff = sum(
            (p1 - p2).abs().sum().item()
            for p1, p2 in zip(pf_rtb.parameters(), pf_tpcl.parameters())
        )

    log["loss_rtb"].append(loss_rtb.item())
    log["loss_tpcl"].append(loss_tpcl.item())
    log["logZ"].append(logZ_val)
    log["v_soft_s0"].append(v_soft_val)
    log["weight_diff"].append(weight_diff)

    # Verify the exact relationship at every step.
    assert torch.allclose(
        loss_tpcl.detach(), ALPHA**2 * loss_rtb.detach(), atol=1e-5
    ), f"Step {step}: loss_tpcl={loss_tpcl.item():.6f} != alpha^2*loss_rtb={ALPHA**2 * loss_rtb.item():.6f}"

print(f"Training complete. All {N_ITERATIONS} steps satisfy L_TPCL = alpha^2 * L_RTB.")
print(f"Final weight difference: {log['weight_diff'][-1]:.2e}")
print(f"Final logZ (RTB): {log['logZ'][-1]:.4f}")
print(f"Final v_soft_s0 / alpha (Trust-PCL): {log['v_soft_s0'][-1] / ALPHA:.4f}")

Training complete. All 5000 steps satisfy L_TPCL = alpha^2 * L_RTB.
Final weight difference: 0.00e+00
Final logZ (RTB): -0.6750
Final v_soft_s0 / alpha (Trust-PCL): -0.6750


## 4. Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# --- Panel 1: Loss curves ---
ax = axes[0]
ax.plot(log["loss_rtb"], label="RTB loss", alpha=0.5, linewidth=2)
ax.plot([l / ALPHA**2 for l in log["loss_tpcl"]], label="Trust-PCL loss / α²",
        linestyle="--", alpha=0.5)
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("Loss equivalence: L_TPCL = α² · L_RTB")
ax.legend()
ax.set_yscale("log")

# --- Panel 2: logZ vs v_soft_s0 / alpha ---
ax = axes[1]
ax.plot(log["logZ"], label="logZ (RTB)", alpha=0.5, linewidth=2)
ax.plot([v / ALPHA for v in log["v_soft_s0"]], label="v_soft_s0 / α (Trust-PCL)",
        linestyle="--", alpha=0.5)
ax.set_xlabel("Training step")
ax.set_ylabel("Value")
ax.set_title("Learned scalar: v_soft_s0 = α · logZ")
ax.legend()

# --- Panel 3: Weight difference ---
ax = axes[2]
ax.plot(log["weight_diff"])
ax.set_xlabel("Training step")
ax.set_ylabel("Sum of |weight differences|")
ax.set_title("Weight divergence (should be 0)")
ax.ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))

plt.tight_layout()
plt.show()

## 5. What this demonstrates

The three plots above should show:

1. **Left**: The RTB and Trust-PCL/α² loss curves overlap perfectly — they are the
   same optimization objective up to a constant scaling factor.

2. **Middle**: The RTB `logZ` and Trust-PCL `v_soft_s0 / α` track each other
   exactly — they are the same learned quantity expressed in different units.
   Note: `logZ` estimates `log Σ_x p_prior(x) · r(x)^β`, which depends on both the
   prior and β. This is generally **not** the same as the environment's
   `log Σ_x r(x)`. What matters for the equivalence is that both GFlowNets
   learn the same value.

3. **Right**: The total weight difference between the two independently-trained
   networks stays at exactly zero — they follow identical optimization trajectories.

### The takeaway

RTB (GFlowNet) and Trust-PCL (RL) are **not analogous** — they are **identical**.
The same neural network weights are updated by the same gradients (up to a constant
that is absorbed by the learning rate). The only difference is vocabulary:

- A GFlowNet researcher says "posterior policy", "prior", "logZ", "beta"
- An RL researcher says "policy", "reference policy", "soft value", "alpha"

This equivalence (Deleu et al. 2025, [arXiv:2509.01632](https://arxiv.org/abs/2509.01632))
means that insights from entropy-regularized RL (soft actor-critic, Trust-PCL,
KL-regularized policy optimization) transfer directly to GFlowNet training, and vice versa.

### Technical note: why SGD, not Adam?

We use SGD because Adam's adaptive step size normalizes by the second moment of
gradients. Since Trust-PCL gradients are α² times larger, the denominator
(`sqrt(v) + ε`) differs between the two optimizers, and the `ε` term prevents
exact cancellation. With plain SGD, `update = lr · grad`, so the α² in the
gradient cancels exactly with `1/α²` in the learning rate.